## Printing Initial Classes

Before model training, let's examine the classes in our dataset to understand the classification task.

In [ ]:
# Print the unique classes in the target variable
print("Classes in the dataset:")
unique_classes = y.unique()
print(unique_classes)
print(f"Number of classes: {len(unique_classes)}")

# Class distribution
class_distribution = y.value_counts()
print("\nClass distribution:")
print(class_distribution)

# Visualize class distribution
plt.figure(figsize=(10, 6))
sns.countplot(y=y)
plt.title('Class Distribution')
plt.xlabel('Class')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Cross-Validation Implementation

Implementing k-fold cross-validation to ensure robust model evaluation.

In [ ]:
from sklearn.model_selection import KFold, cross_val_score, StratifiedKFold

# Define models to evaluate with cross-validation
cv_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(),
    'SVM': SVC(probability=True),
    'Naive Bayes': MultinomialNB()
}

# Setting up stratified k-fold cross validation
n_folds = 5
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

# Dictionary to store cross-validation results
cv_results = {}

for name, model in cv_models.items():
    # Perform cross-validation
    cv_scores = cross_val_score(model, X_train_tfidf, y_train, cv=skf, scoring='accuracy')
    
    # Store results
    cv_results[name] = {
        'mean_accuracy': cv_scores.mean(),
        'std_accuracy': cv_scores.std(),
        'all_folds': cv_scores
    }
    
    print(f"{name} - Cross-Validation Results:")
    print(f"Mean Accuracy: {cv_scores.mean():.4f}")
    print(f"Standard Deviation: {cv_scores.std():.4f}")
    print(f"All Folds: {cv_scores}")
    print("-" * 50)

# Visualize cross-validation results
plt.figure(figsize=(12, 6))
models = list(cv_results.keys())
mean_scores = [cv_results[model]['mean_accuracy'] for model in models]
std_scores = [cv_results[model]['std_accuracy'] for model in models]

bars = plt.bar(models, mean_scores, yerr=std_scores, capsize=10, alpha=0.7)
plt.title('Cross-Validation Accuracy Across Models')
plt.ylabel('Mean Accuracy')
plt.xlabel('Models')
plt.xticks(rotation=45)
plt.ylim(0.5, 1.0)  # Adjust as needed
plt.tight_layout()

# Add actual values on top of bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.01,
            f'{height:.3f}', ha='center', va='bottom')

plt.show()

## Hyperparameter Tuning with Grid Search

Implementing Grid Search to find the optimal hyperparameters for our models.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score

# Define the parameter grids for each model
param_grids = {
    'Logistic Regression': {
        'model': LogisticRegression(max_iter=1000),
        'params': {
            'C': [0.1, 1, 10, 100],
            'solver': ['liblinear', 'lbfgs'],
            'penalty': ['l1', 'l2']
        }
    },
    'Random Forest': {
        'model': RandomForestClassifier(),
        'params': {
            'n_estimators': [50, 100, 200],
            'max_depth': [None, 10, 20, 30],
            'min_samples_split': [2, 5, 10]
        }
    },
    'SVM': {
        'model': SVC(probability=True),
        'params': {
            'C': [0.1, 1, 10, 100],
            'kernel': ['linear', 'rbf'],
            'gamma': ['scale', 'auto', 0.1, 1]
        }
    }
}

# Scoring metrics
scoring = {
    'accuracy': make_scorer(accuracy_score),
    'precision': make_scorer(precision_score, average='weighted'),
    'recall': make_scorer(recall_score, average='weighted'),
    'f1': make_scorer(f1_score, average='weighted')
}

# Perform Grid Search for each model
best_models = {}
grid_search_results = {}

for name, config in param_grids.items():
    print(f"Performing Grid Search for {name}...")
    
    grid_search = GridSearchCV(
        estimator=config['model'],
        param_grid=config['params'],
        cv=5,
        scoring=scoring,
        refit='accuracy',
        verbose=1,
        n_jobs=-1
    )
    
    grid_search.fit(X_train_tfidf, y_train)
    
    # Store the best model
    best_models[name] = grid_search.best_estimator_
    
    # Store grid search results
    grid_search_results[name] = {
        'best_params': grid_search.best_params_,
        'best_score': grid_search.best_score_,
        'cv_results': grid_search.cv_results_
    }
    
    print(f"\nBest parameters for {name}:")
    print(grid_search.best_params_)
    print(f"Best cross-validation score: {grid_search.best_score_:.4f}")
    print("-" * 70)

# Summary of hyperparameter tuning results
print("\nSummary of Hyperparameter Tuning Results:")
for name, results in grid_search_results.items():
    print(f"{name}:")
    print(f"  Best Parameters: {results['best_params']}")
    print(f"  Best CV Accuracy: {results['best_score']:.4f}")

## Comparing CM and ROC between Training and Testing

Generate and compare confusion matrices and ROC curves for both training and testing datasets to identify potential overfitting.

In [ ]:
from sklearn.metrics import confusion_matrix, roc_curve, auc, RocCurveDisplay
import itertools

def plot_confusion_matrix(cm, classes, title='Confusion Matrix', cmap=plt.cm.Blues):
    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)
    
    # Add text annotations
    fmt = 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")
    
    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()

def plot_roc_curves(model, X_data, y_data, model_name, dataset_type):
    y_probs = model.predict_proba(X_data)
    
    # For binary classification
    if len(np.unique(y_data)) == 2:
        fpr, tpr, _ = roc_curve(y_data, y_probs[:, 1])
        roc_auc = auc(fpr, tpr)
        
        plt.figure(figsize=(8, 6))
        plt.plot(fpr, tpr, lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
        plt.plot([0, 1], [0, 1], 'k--', lw=2)
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(f'ROC Curve for {model_name} - {dataset_type}')
        plt.legend(loc="lower right")
        plt.show()
    
    # For multi-class classification
    else:
        # One-vs-Rest approach for ROC
        n_classes = len(np.unique(y_data))
        y_test_bin = label_binarize(y_data, classes=np.unique(y_data))
        
        fpr = {}
        tpr = {}
        roc_auc = {}
        
        plt.figure(figsize=(10, 8))
        
        for i in range(n_classes):
            fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_probs[:, i])
            roc_auc[i] = auc(fpr[i], tpr[i])
            plt.plot(fpr[i], tpr[i], lw=2,
                     label=f'Class {i} (area = {roc_auc[i]:.2f})')
        
        plt.plot([0, 1], [0, 1], 'k--', lw=2)
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(f'ROC Curve for {model_name} - {dataset_type}')
        plt.legend(loc="lower right")
        plt.show()

# Compare best model from grid search
for name, model in best_models.items():
    print(f"\n---- Analysis for {name} ----")
    
    # Predictions on train and test sets
    y_train_pred = model.predict(X_train_tfidf)
    y_test_pred = model.predict(X_test_tfidf)
    
    # Confusion matrices
    cm_train = confusion_matrix(y_train, y_train_pred)
    cm_test = confusion_matrix(y_test, y_test_pred)
    
    # Plot confusion matrices
    plot_confusion_matrix(cm_train, classes=np.unique(y), 
                         title=f'Confusion Matrix - {name} (Training)')
    plt.show()
    
    plot_confusion_matrix(cm_test, classes=np.unique(y),
                         title=f'Confusion Matrix - {name} (Testing)')
    plt.show()
    
    # Plot ROC curves
    print("ROC Curves for Training Data:")
    plot_roc_curves(model, X_train_tfidf, y_train, name, "Training")
    
    print("ROC Curves for Testing Data:")
    plot_roc_curves(model, X_test_tfidf, y_test, name, "Testing")
    
    # Calculate metrics for both sets
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
    
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    
    train_prec = precision_score(y_train, y_train_pred, average='weighted')
    test_prec = precision_score(y_test, y_test_pred, average='weighted')
    
    train_rec = recall_score(y_train, y_train_pred, average='weighted')
    test_rec = recall_score(y_test, y_test_pred, average='weighted')
    
    train_f1 = f1_score(y_train, y_train_pred, average='weighted')
    test_f1 = f1_score(y_test, y_test_pred, average='weighted')
    
    # Print comparison of metrics
    print("\nPerformance Comparison:")
    metrics_df = pd.DataFrame({
        'Training': [train_acc, train_prec, train_rec, train_f1],
        'Testing': [test_acc, test_prec, test_rec, test_f1]
    }, index=['Accuracy', 'Precision', 'Recall', 'F1 Score'])
    
    print(metrics_df)
    
    # Visualize train vs test metrics
    metrics_df.plot(kind='bar', figsize=(10, 6))
    plt.title(f'Training vs Testing Metrics for {name}')
    plt.ylabel('Score')
    plt.ylim(0, 1.1)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.xticks(rotation=0)
    
    for i, col in enumerate(metrics_df.columns):
        for j, val in enumerate(metrics_df[col]):
            plt.text(j+(i*0.25), val + 0.02, f'{val:.3f}', 
                     ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()

## Analysis of Overfitting, Underfitting, and Data Leakage

Let's analyze our models for potential issues like overfitting, underfitting, and data leakage.

In [ ]:
# Function to analyze model performance for overfitting/underfitting
def analyze_model_fit(name, train_metrics, test_metrics):
    train_acc = train_metrics['accuracy']
    test_acc = test_metrics['accuracy']
    
    print(f"\nAnalysis for {name}:")
    
    # Calculate the difference between training and testing accuracy
    diff = train_acc - test_acc
    
    if diff > 0.1:  # Threshold can be adjusted
        print(f"❌ Potential OVERFITTING detected: Training accuracy is {diff:.4f} higher than testing accuracy")
        print("   The model performs significantly better on training data than on unseen data.")
        print("   Recommendations:")
        print("   - Implement regularization")
        print("   - Reduce model complexity")
        print("   - Get more training data")
        print("   - Use early stopping for neural networks")
    elif test_acc > train_acc:
        print(f"⚠️ Unusual pattern: Testing accuracy is {abs(diff):.4f} higher than training accuracy")
        print("   This could indicate:")
        print("   - Data leakage: testing data might be influencing the training process")
        print("   - Sampling issues: test set might be easier to predict than training set")
        print("   - Random variation: especially if the difference is small")
    elif train_acc < 0.6 and test_acc < 0.6:  # Threshold can be adjusted
        print(f"❌ Potential UNDERFITTING detected: Both training ({train_acc:.4f}) and testing ({test_acc:.4f}) accuracy are low")
        print("   The model isn't capturing the underlying patterns in the data.")
        print("   Recommendations:")
        print("   - Increase model complexity")
        print("   - Add more features")
        print("   - Try different algorithms")
        print("   - Reduce regularization")
    else:
        print(f"✅ Model appears to be well-balanced: Training accuracy: {train_acc:.4f}, Testing accuracy: {test_acc:.4f}")
        print(f"   The difference is {abs(diff):.4f}, which is within an acceptable range.")
    
    # Learning curves can help visualize overfitting/underfitting
    if abs(diff) > 0.05:
        print("\nRecommendation: Generate learning curves to further diagnose fitting issues.")

# Analyze each of our best models
for name, model in best_models.items():
    # Get predictions
    y_train_pred = model.predict(X_train_tfidf)
    y_test_pred = model.predict(X_test_tfidf)
    
    # Calculate metrics
    train_metrics = {
        'accuracy': accuracy_score(y_train, y_train_pred),
        'precision': precision_score(y_train, y_train_pred, average='weighted'),
        'recall': recall_score(y_train, y_train_pred, average='weighted'),
        'f1': f1_score(y_train, y_train_pred, average='weighted')
    }
    
    test_metrics = {
        'accuracy': accuracy_score(y_test, y_test_pred),
        'precision': precision_score(y_test, y_test_pred, average='weighted'),
        'recall': recall_score(y_test, y_test_pred, average='weighted'),
        'f1': f1_score(y_test, y_test_pred, average='weighted')
    }
    
    # Analyze for overfitting/underfitting
    analyze_model_fit(name, train_metrics, test_metrics)

# Generate learning curves for a selected model (e.g., the best one)
from sklearn.model_selection import learning_curve

def plot_learning_curve(estimator, title, X, y, cv=5, n_jobs=-1, train_sizes=np.linspace(0.1, 1.0, 5)):
    plt.figure(figsize=(10, 6))
    plt.title(title)
    plt.xlabel("Training examples")
    plt.ylabel("Score")
    
    train_sizes, train_scores, test_scores = learning_curve(
        estimator, X, y, cv=cv, n_jobs=n_jobs, train_sizes=train_sizes,
        scoring="accuracy", shuffle=True, random_state=42)
    
    train_scores_mean = np.mean(train_scores, axis=1)
    train_scores_std = np.std(train_scores, axis=1)
    test_scores_mean = np.mean(test_scores, axis=1)
    test_scores_std = np.std(test_scores, axis=1)
    
    plt.grid()
    plt.fill_between(train_sizes, train_scores_mean - train_scores_std,
                     train_scores_mean + train_scores_std, alpha=0.1, color="r")
    plt.fill_between(train_sizes, test_scores_mean - test_scores_std,
                     test_scores_mean + test_scores_std, alpha=0.1, color="g")
    plt.plot(train_sizes, train_scores_mean, 'o-', color="r", label="Training score")
    plt.plot(train_sizes, test_scores_mean, 'o-', color="g", label="Cross-validation score")
    plt.legend(loc="best")
    
    return plt

# Get the best model from our grid search
best_model_name = max(grid_search_results, key=lambda x: grid_search_results[x]['best_score'])
best_model = best_models[best_model_name]

print(f"\nGenerating learning curve for {best_model_name}...")
plot_learning_curve(
    best_model, 
    f"Learning Curves for {best_model_name}", 
    X_train_tfidf, y_train, 
    cv=5
)
plt.show()

# Check for data leakage indicators
print("\n=== Checking for potential data leakage ===")
print("1. Unusually high testing performance")
for name, model in best_models.items():
    y_test_pred = model.predict(X_test_tfidf)
    test_acc = accuracy_score(y_test, y_test_pred)
    if test_acc > 0.95:  # Adjust threshold as needed
        print(f"⚠️ {name} has suspiciously high test accuracy: {test_acc:.4f}")
        print("   This may indicate data leakage if not expected from domain knowledge")

print("\n2. Feature importance analysis (for applicable models)")
if 'Random Forest' in best_models:
    rf_model = best_models['Random Forest']
    
    # Get feature importance
    feature_importances = rf_model.feature_importances_
    
    # Get feature names 
    try:
        feature_names = X_train_tfidf.get_feature_names_out()
        
        # Create DataFrame for feature importance
        feature_importance_df = pd.DataFrame({
            'Feature': feature_names,
            'Importance': feature_importances
        })
        
        # Sort by importance
        feature_importance_df = feature_importance_df.sort_values('Importance', ascending=False).head(20)
        
        # Plot
        plt.figure(figsize=(10, 8))
        sns.barplot(x='Importance', y='Feature', data=feature_importance_df)
        plt.title('Top 20 Important Features')
        plt.tight_layout()
        plt.show()
        
        # Check if there are any suspiciously dominant features
        max_importance = feature_importance_df['Importance'].max()
        if max_importance > 0.5:
            print(f"⚠️ Feature '{feature_importance_df.iloc[0]['Feature']}' has unusually high importance: {max_importance:.4f}")
            print("   This could indicate data leakage if this feature shouldn't be so predictive")
    except:
        print("Couldn't extract feature names from TF-IDF vectorizer")

## Loss Graphs and Training History for CNN and FNN Models

Adding visualization of training progress for the neural network models.

In [ ]:
# This cell assumes CNN and FNN models have been trained with history objects
# If not, you'll need to retrain the models with history capture

# Function to plot training history
def plot_training_history(history, title='Model Training History'):
    plt.figure(figsize=(12, 5))
    
    # Plot loss
    plt.subplot(1, 2, 1)
    plt.plot(history.history['loss'], label='Training Loss')
    if 'val_loss' in history.history:
        plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title(f'{title} - Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    
    # Plot accuracy
    plt.subplot(1, 2, 2)
    plt.plot(history.history['accuracy'], label='Training Accuracy')
    if 'val_accuracy' in history.history:
        plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title(f'{title} - Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    
    plt.tight_layout()
    plt.show()

# If you need to retrain the CNN model with history capture
# First, check if you have a CNN model already defined
try:
    # Create a simple CNN model for text classification
    from tensorflow.keras.preprocessing.text import Tokenizer
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout
    
    # Tokenize and pad the text data
    max_words = 10000
    max_sequence_length = 100
    
    tokenizer = Tokenizer(num_words=max_words)
    tokenizer.fit_on_texts(X_train)
    
    X_train_seq = tokenizer.texts_to_sequences(X_train)
    X_test_seq = tokenizer.texts_to_sequences(X_test)
    
    X_train_padded = pad_sequences(X_train_seq, maxlen=max_sequence_length)
    X_test_padded = pad_sequences(X_test_seq, maxlen=max_sequence_length)
    
    # Prepare labels for CNN/FNN
    from sklearn.preprocessing import LabelEncoder
    label_encoder = LabelEncoder()
    y_train_encoded = label_encoder.fit_transform(y_train)
    y_test_encoded = label_encoder.transform(y_test)
    
    # Convert to categorical if needed for multi-class
    from tensorflow.keras.utils import to_categorical
    num_classes = len(np.unique(y_train))
    
    if num_classes > 2:  # Multi-class case
        y_train_cat = to_categorical(y_train_encoded, num_classes)
        y_test_cat = to_categorical(y_test_encoded, num_classes)
    else:  # Binary case
        y_train_cat = y_train_encoded
        y_test_cat = y_test_encoded
    
    # CNN Model
    embedding_dim = 100
    cnn_model = Sequential([
        Embedding(max_words, embedding_dim, input_length=max_sequence_length),
        Conv1D(128, 5, activation='relu'),
        GlobalMaxPooling1D(),
        Dense(64, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='sigmoid' if num_classes == 2 else 'softmax')
    ])
    
    # Compile model
    cnn_model.compile(
        optimizer='adam',
        loss='binary_crossentropy' if num_classes == 2 else 'categorical_crossentropy',
        metrics=['accuracy']
    )
    
    # Train with validation split and capture history
    cnn_history = cnn_model.fit(
        X_train_padded, y_train_cat,
        validation_split=0.2,
        epochs=10,
        batch_size=32,
        verbose=1
    )
    
    # Plot CNN training history
    plot_training_history(cnn_history, title='CNN Model')
    
    # FNN Model (Feedforward Neural Network)
    from tensorflow.keras.layers import Flatten, Dense, Dropout, Input
    from tensorflow.keras.layers import BatchNormalization
    
    # For FNN, we'll use TF-IDF features instead of embeddings
    fnn_model = Sequential([
        Input(shape=(X_train_tfidf.shape[1],)),  # Input shape from TF-IDF
        Dense(256, activation='relu'),
        BatchNormalization(),
        Dropout(0.5),
        Dense(128, activation='relu'),
        Dropout(0.3),
        Dense(num_classes, activation='sigmoid' if num_classes == 2 else 'softmax')
    ])
    
    # Compile model
    fnn_model.compile(
        optimizer='adam',
        loss='binary_crossentropy' if num_classes == 2 else 'categorical_crossentropy',
        metrics=['accuracy']
    )
    
    # Train with validation split and capture history
    fnn_history = fnn_model.fit(
        X_train_tfidf.toarray(), y_train_cat,
        validation_split=0.2,
        epochs=15,
        batch_size=32,
        verbose=1
    )
    
    # Plot FNN training history
    plot_training_history(fnn_history, title='FNN Model')
    
    # Compare CNN and FNN performances
    cnn_train_loss, cnn_train_acc = cnn_model.evaluate(X_train_padded, y_train_cat, verbose=0)
    cnn_test_loss, cnn_test_acc = cnn_model.evaluate(X_test_padded, y_test_cat, verbose=0)
    
    fnn_train_loss, fnn_train_acc = fnn_model.evaluate(X_train_tfidf.toarray(), y_train_cat, verbose=0)
    fnn_test_loss, fnn_test_acc = fnn_model.evaluate(X_test_tfidf.toarray(), y_test_cat, verbose=0)
    
    # Print comparison
    print("\n=== Neural Network Models Comparison ===")
    comparison_df = pd.DataFrame({
        'CNN': [cnn_train_acc, cnn_test_acc, cnn_train_loss, cnn_test_loss],
        'FNN': [fnn_train_acc, fnn_test_acc, fnn_train_loss, fnn_test_loss]
    }, index=['Training Accuracy', 'Testing Accuracy', 'Training Loss', 'Testing Loss'])
    
    print(comparison_df)
    
    # Visualize comparison
    plt.figure(figsize=(10, 6))
    comparison_df.loc[['Training Accuracy', 'Testing Accuracy']].plot(kind='bar')
    plt.title('CNN vs FNN Accuracy Comparison')
    plt.ylabel('Accuracy')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.ylim(0, 1.1)
    
    for i, v in enumerate(comparison_df.loc['Training Accuracy']):
        plt.text(i-0.15, v+0.02, f"{v:.3f}", color='black', fontweight='bold')
    for i, v in enumerate(comparison_df.loc['Testing Accuracy']):
        plt.text(i+0.05, v+0.02, f"{v:.3f}", color='black', fontweight='bold')
        
    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f"Error in neural network training: {e}")
    print("If there was an error, please ensure TensorFlow is installed and your dataset is properly preprocessed.")
    print("You may need to adjust the model architecture based on your specific dataset.")

## Print Misclassified Examples

Let's examine misclassified examples from our best model to understand error patterns.

In [ ]:
# Identify the best model overall
best_model_name = max(grid_search_results, key=lambda x: grid_search_results[x]['best_score'])
best_model = best_models[best_model_name]
print(f"Analyzing misclassifications for the best model: {best_model_name}")

# Get predictions
y_test_pred = best_model.predict(X_test_tfidf)

# Find misclassified examples
misclassified_indices = np.where(y_test != y_test_pred)[0]
num_misclassified = len(misclassified_indices)

print(f"Total number of misclassified examples: {num_misclassified} out of {len(y_test)} test samples")
print(f"Misclassification rate: {num_misclassified / len(y_test):.4f}")

# Display some misclassified examples (up to 10)
if num_misclassified > 0:
    print("\nSample of misclassified examples:")
    
    # Store original test data and indices
    X_test_data = X_test.reset_index(drop=True)
    y_test_data = y_test.reset_index(drop=True)
    
    # Number of examples to show
    n_examples = min(10, num_misclassified)
    
    for i, idx in enumerate(misclassified_indices[:n_examples]):
        true_label = y_test_data[idx]
        pred_label = y_test_pred[idx]
        
        print(f"\nMisclassification {i+1}:")
        print(f"Text: {X_test_data[idx][:200]}...")  # Show first 200 chars
        print(f"True label: {true_label}")
        print(f"Predicted label: {pred_label}")
        
        # If the model supports probability scores, show them
        if hasattr(best_model, "predict_proba"):
            proba = best_model.predict_proba(X_test_tfidf[idx])
            classes = best_model.classes_
            
            # Get top 3 predictions with probabilities
            top_indices = np.argsort(proba[0])[::-1][:3]
            top_classes = [classes[i] for i in top_indices]
            top_probas = [proba[0][i] for i in top_indices]
            
            print("Top predictions with probabilities:")
            for cls, prob in zip(top_classes, top_probas):
                print(f"  {cls}: {prob:.4f}")

## Final Performance Summary and Recommendations

Let's create a comprehensive summary of all model performances and make recommendations.

In [ ]:
# Compile results from all models (traditional ML and neural networks)
all_models_results = {}

# Add results from traditional ML models
for name, model in best_models.items():
    # Training metrics
    y_train_pred = model.predict(X_train_tfidf)
    train_acc = accuracy_score(y_train, y_train_pred)
    train_f1 = f1_score(y_train, y_train_pred, average='weighted')
    
    # Testing metrics
    y_test_pred = model.predict(X_test_tfidf)
    test_acc = accuracy_score(y_test, y_test_pred)
    test_f1 = f1_score(y_test, y_test_pred, average='weighted')
    
    all_models_results[name] = {
        'train_accuracy': train_acc,
        'test_accuracy': test_acc,
        'train_f1': train_f1,
        'test_f1': test_f1,
        'difference': train_acc - test_acc
    }

# Add results from neural networks if available
try:
    # For CNN
    if 'cnn_train_acc' in locals() and 'cnn_test_acc' in locals():
        all_models_results['CNN'] = {
            'train_accuracy': cnn_train_acc,
            'test_accuracy': cnn_test_acc,
            'train_f1': None,  # F1 not directly calculated in Keras evaluation
            'test_f1': None,
            'difference': cnn_train_acc - cnn_test_acc
        }
    
    # For FNN
    if 'fnn_train_acc' in locals() and 'fnn_test_acc' in locals():
        all_models_results['FNN'] = {
            'train_accuracy': fnn_train_acc,
            'test_accuracy': fnn_test_acc,
            'train_f1': None,  # F1 not directly calculated in Keras evaluation
            'test_f1': None,
            'difference': fnn_train_acc - fnn_test_acc
        }
except:
    print("Neural network results not available")

# Create a DataFrame for easy comparison
results_df = pd.DataFrame(all_models_results).T
results_df = results_df.sort_values('test_accuracy', ascending=False)

print("=== Final Performance Summary ===")
print(results_df)

# Identify the best performing model on test data
best_model_name = results_df['test_accuracy'].idxmax()
best_accuracy = results_df.loc[best_model_name, 'test_accuracy']

print(f"\nBest performing model: {best_model_name} with test accuracy of {best_accuracy:.4f}")

# Check for overfitting across all models
overfit_models = results_df[results_df['difference'] > 0.1]
if not overfit_models.empty:
    print("\nModels with potential overfitting:")
    print(overfit_models)

# Final recommendations
print("\n=== Recommendations ===")
print(f"1. The {best_model_name} model performs best on unseen data with {best_accuracy:.4f} accuracy.")

if best_model_name in grid_search_results:
    print(f"2. Best hyperparameters for {best_model_name}:")
    for param, value in grid_search_results[best_model_name]['best_params'].items():
        print(f"   - {param}: {value}")

if not overfit_models.empty:
    print("3. Some models show signs of overfitting. Consider:")
    print("   - Increasing regularization")
    print("   - Collecting more training data")
    print("   - Using simpler models")

print("4. Further steps to improve model performance:")
print("   - Feature engineering: try different text representations")
print("   - Ensemble methods: combine predictions from multiple models")
print("   - Advanced preprocessing: stemming, lemmatization, etc.")
print("   - External knowledge integration: domain-specific lexicons or embeddings")

# Create a final visualization of model performances
plt.figure(figsize=(12, 8))

# Create a grouped bar chart
bar_width = 0.35
index = np.arange(len(results_df))

plt.bar(index, results_df['train_accuracy'], bar_width, label='Training Accuracy', color='skyblue')
plt.bar(index + bar_width, results_df['test_accuracy'], bar_width, label='Testing Accuracy', color='lightcoral')

# Add labels and title
plt.xlabel('Model')
plt.ylabel('Accuracy')
plt.title('Training vs Testing Accuracy Across All Models')
plt.xticks(index + bar_width/2, results_df.index, rotation=45, ha='right')
plt.legend()
plt.ylim(0.5, 1.0)  # Adjust as needed
plt.tight_layout()

# Add value labels on bars
for i, (train_acc, test_acc) in enumerate(zip(results_df['train_accuracy'], results_df['test_accuracy'])):
    plt.text(i, train_acc + 0.01, f'{train_acc:.3f}', ha='center', va='bottom', fontsize=9)
    plt.text(i + bar_width, test_acc + 0.01, f'{test_acc:.3f}', ha='center', va='bottom', fontsize=9)

plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()